# 3.3. 기본 CF 알고리즘

In [14]:
# 데이터 읽어오기(user, item, data)
import os
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split

base_src = Path.cwd().parent / 'data'

# user 데이터
u_user_src = os.path.join(base_src,'u.user')
u_cols = ['user_id','age','sex','occupation','zip_code']
users = pd.read_csv(u_user_src,
	sep = '|',
        names = u_cols,
        encoding = 'latin-1')
users = users.set_index('user_id')

# movie 데이터
u_item_src = os.path.join(base_src,'u.item')
i_cols = ['movie_id','title','release date','video release date','IMDB URL','unknown','Action','Adventure','Animation','Children\'s','Comedy','Crime','Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery','Romance','Sci-Fi','Thriller','War','Western']
movies = pd.read_csv(u_item_src,
	    sep='|',
            names=i_cols,
            encoding='latin-1')
movies = movies[['movie_id','title']]

# rating 데이터
u_data_src = os.path.join(base_src,'u.data')
r_cols = ['user_id','movie_id','rating','timestamp']
ratings = pd.read_csv(u_data_src,
        sep = '\t',
        names = r_cols,
        encoding='latin-1')
ratings = ratings.drop('timestamp',axis=1)

# RMSE 함수
def RMSE(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2))

# score(RMSE) 계산하는 함수
def score(model):
    id_pairs = zip(x_test['user_id'], x_test['movie_id'])
    y_pred = np.array([model(user, movie) for (user, movie) in id_pairs])
    y_true = np.array(x_test['rating'])
    return RMSE(y_true, y_pred)

# 데이터 셋
x = ratings.copy()
y = ratings['user_id']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y)
ratings_matrix = x_train.pivot(index='user_id',columns = 'movie_id',values='rating')


In [16]:
# 코사인 유사도 계산

from sklearn.metrics.pairwise import cosine_similarity

matrix_copy = ratings_matrix.copy().fillna(0)

user_similarity = cosine_similarity(matrix_copy, matrix_copy)
user_similarity = pd.DataFrame(user_similarity, index=ratings_matrix.index, columns=ratings_matrix.index)

# 주어진 영화의 가중평균 rating을 계산하는 함수

def CF_simple(user_id, movie_id):
    if movie_id in ratings_matrix.columns:
        sim_scores = user_similarity[user_id].copy()
        movie_ratings = ratings_matrix[movie_id].copy()
        none_rating_idx = movie_ratings[movie_ratings.isnull()].index
        movie_ratings = movie_ratings.dropna()
        sim_scores = sim_scores.drop(none_rating_idx)
        return np.dot(sim_scores, movie_ratings) / sim_scores.sum()
    else:
        return 3.0

score(CF_simple)


np.float64(1.0164996704904246)